In [11]:
import os
import json
import pandas as pd
from types import SimpleNamespace

from datasets import load_dataset

In [4]:
ds = load_dataset("abisee/cnn_dailymail", "3.0.0")['test']
df = ds.to_pandas()
df.head(2)

Generating test split: 100%|██████████| 11490/11490 [00:00<00:00, 18786.96 examples/s]


,article,highlights,id
0,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,f001ec5c4704938247d27a44948eebb37ae98d01
1,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",230c522854991d053fe98a718b1defa077a8efef


In [9]:
df["article_len"] = df["article"].apply(len)
len(df[df["article_len"] < 4000])

6730

In [7]:
sample = "(CNN)Five Americans who were monitored for three weeks at an Omaha, Nebraska, hospital after being exposed to Ebola in West Africa have been released, a Nebraska Medicine spokesman said in an email Wednesday. One of the five had a heart-related issue on Saturday and has been discharged but hasn't left the area, Taylor Wilson wrote. The others have already gone home. They were exposed to Ebola in Sierra Leone in March, but none developed the deadly virus. They are clinicians for Partners in Health, a Boston-based aid group. They all had contact with a colleague who was diagnosed with the disease and is being treated at the National Institutes of Health in Bethesda, Maryland. As of Monday, that health care worker is in fair condition. The Centers for Disease Control and Prevention in Atlanta has said the last of 17 patients who were being monitored are expected to be released by Thursday. More than 10,000 people have died in a West African epidemic of Ebola that dates to December 2013, according to the World Health Organization. Almost all the deaths have been in Guinea, Liberia and Sierra Leone. Ebola is spread by direct contact with the bodily fluids of an infected person."
len(sample)

1191

In [15]:
os.makedirs("data", exist_ok=True)
df[df["article_len"] < 4000][["id", "article", "highlights"]].to_json("data/cnn_dailymail.jsonl", orient="records", lines=True)

In [18]:
selected_df = pd.read_json("data/cnn_dailymail.jsonl", lines=True)
selected_df['article'] = selected_df['article'].str.replace(r'^\([^)]*\)\s*', '', regex=True)
selected_df.to_json("data/cnn_dailymail.jsonl", orient="records", lines=True)

In [24]:
selected_df["safe_prompt"] = True
selected_df.rename(columns={"article": "prompt"}, inplace=True)
to_prod_df = selected_df.sample(n=100)
to_prod_df.to_json("data_final/vectara_v1.jsonl", orient="records", lines=True)
to_prod_df.to_json("data/vectara_v1.jsonl", orient="records", lines=True)
to_prod_df.head(2)

,id,prompt,highlights,safe_prompt
3663,56c80675f3fc3c8405b283e336592e8fe4eb8857,A boxing fight had to be halted after a fighte...,"Marvin ""Papi Gallo"" Jones and Ramon Luis Nicol...",True
2239,c61bca1995c65675d572f131b24bfd83acb6bdfa,Bayern Munich sporting director Matthias Samme...,Bayern director Matthias Sammer is worried abo...,True
